# v3.1 training and validation — fresh notebook
Copied from successful rsna-hpo-v3-1-d2-resnet50-fold0 version2. Same attachments and runtime; v3.1 controls validation, Steven-v4 logged only.


In [ ]:
#!/usr/bin/env python3
"""Kaggle runner for the HPO-v3.1-D2 label comparison.

It keeps the HPO-v1 +281 ResNet-50 recipe fixed and changes only the train
label probabilities. Fold-0 IDs remain the same 825 studies; v3.1-D2 controls validation and Steven-v4 is an audit reference.
The global study batch stays at 8 and uses one T4, matching the reference.
"""

from __future__ import annotations

print("STARTUP: fresh copy of successful v3.1 ResNet50 notebook; trainval experiment", flush=True)

import csv
import hashlib
import importlib.util
import json
import sys
from collections import OrderedDict
from pathlib import Path

import os
import time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch


TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
LABEL_SHA256 = "38367d62eb9e7c198b69d88e2b9ad419e00fea9d1211b98feae45cd8a386f92d"
CONFIG_SHA256 = "31125adb466de6490c0797687401f9608b7b92bf769f0d6b1a40b4b690c7568c"
BASE_FOLDS_SHA256 = "2ed70719f9423d3a0c3b9ac424841f257d57dda1b9dfd72b26805de8eb4bb6ba"
ADDED_IDS_SHA256 = "82c45331e89e4d9d78636c2d798c256d94fe430bd05e64861650da09d4855a36"


def exact_path(candidates: list[Path], description: str) -> Path:
    found = [path for path in candidates if path.is_file()]
    if len(found) != 1:
        raise FileNotFoundError(f"Expected exactly one {description}; found {found}")
    return found[0]


def exact_dir(candidates: list[Path], description: str) -> Path:
    found = [path for path in candidates if path.is_dir()]
    if len(found) != 1:
        raise FileNotFoundError(f"Expected exactly one {description} directory; found {found}")
    return found[0]


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main() -> None:
    input_root = Path("/kaggle/input")
    trainer_input = exact_path([
        input_root / "datasets/tumis001/rsna-cnn-hpo-blend-xla-source-v1/skanee/scripts/train_cnn_baseline.py",
        input_root / "rsna-cnn-hpo-blend-xla-source-v1/skanee/scripts/train_cnn_baseline.py",
    ], "HPO ResNet training script")
    source = trainer_input.parents[1]

    labels_path = exact_path([
        input_root / "datasets/tumis001/rsna-knee-hpo-v31-d2-report-labels/soft_labels_hpo_v3_1_d2_structured.json",
        input_root / "rsna-knee-hpo-v31-d2-report-labels/soft_labels_hpo_v3_1_d2_structured.json",
    ], "HPO-v3.1-D2 provenance candidate labels")
    steven_csv = exact_path([
        input_root / "datasets/stevenleehans/rsna-knee-llm-report-labels/llm_labels_v4_blend.csv",
        input_root / "rsna-knee-llm-report-labels/llm_labels_v4_blend.csv",
    ], "Steven-v4 validation labels")
    cache_dirs = [
        exact_dir([
            input_root / f"datasets/tumis001/rsna-cnn-topk-support-cache-shard-{i}-rehost",
            input_root / f"rsna-cnn-topk-support-cache-shard-{i}-rehost",
            
        ], f"cache shard {i} dataset")
        for i in range(3)
    ]
    expected_cache_files = ["cache_summary.json", "manifest.jsonl"]
    expected_cache_files += [f"slices_{i:03d}.npy" for i in range(4)]
    expected_cache_files += [f"support_{i:03d}.npy" for i in range(4)]
    for shard_index, cache_dir in enumerate(cache_dirs):
        missing = [name for name in expected_cache_files if not (cache_dir / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"Attached cache shard {shard_index} at {cache_dir} is incomplete; missing {missing}"
            )
        print(f"Cache shard {shard_index} resolved: {cache_dir} ({len(expected_cache_files)} required files)",
              flush=True)

    baseline_folds_path = source / "configs/report_folds_v5.csv"
    gold_split_path = source / "configs/gold_split_v1.csv"
    base_config_path = source / "configs/cnn_hpo_v1_blend.json"
    trainer_path = source / "scripts/train_cnn_baseline.py"
    for path in (baseline_folds_path, gold_split_path, base_config_path, trainer_path):
        if not path.is_file():
            raise FileNotFoundError(f"Missing required experiment input: {path}")

    if sha256(labels_path) != LABEL_SHA256:
        raise ValueError("Attached HPO-v3.1-D2 labels do not match the audited candidate")
    if sha256(base_config_path) != CONFIG_SHA256:
        raise ValueError("Attached training config differs from the fixed HPO-v1 recipe")
    if sha256(baseline_folds_path) != BASE_FOLDS_SHA256:
        raise ValueError("Attached baseline fold assignments differ from the fixed experiment split")

    with labels_path.open(encoding="utf-8") as stream:
        labels = json.load(stream)
    if labels.get("targets_in_order") != TARGETS or len(labels.get("labels", [])) != 4349:
        raise ValueError("HPO-v3.1-D2 schema/coverage check failed")
    label_ids = {row["study_id"] for row in labels["labels"]}
    if len(label_ids) != 4349:
        raise ValueError("HPO-v3.1-D2 labels contain duplicate study IDs")

    def read_fold_ids(path: Path) -> dict[str, int]:
        with path.open(newline="", encoding="utf-8") as stream:
            rows = list(csv.DictReader(stream))
        mapping = {row["study_id"]: int(row["fold"]) for row in rows}
        if len(mapping) != len(rows):
            raise ValueError(f"Duplicate study IDs in {path}")
        return mapping

    base_folds = read_fold_ids(baseline_folds_path)
    validation_ids = {study_id for study_id, fold in base_folds.items() if fold == 0}
    added_ids = label_ids - set(base_folds)
    added_hash = hashlib.sha256(("\n".join(sorted(added_ids)) + "\n").encode("utf-8")).hexdigest()
    if (len(base_folds), len(validation_ids), len(added_ids), added_hash) != (
            4068, 825, 281, ADDED_IDS_SHA256):
        raise ValueError("Expected 3,524 train / fixed 825 validation / 281 train-only IDs")
    if not validation_ids <= label_ids:
        raise ValueError("The fixed validation IDs are not all present in HPO-v3.1-D2 labels")
    # Match the known-good Kaggle ResNet50 runner: reconstruct the canonical
    # expanded split from the attached 4,068-study fold file and the 281 IDs
    # added by the HPO label dataset. The source dataset intentionally does
    # not include a second expanded-fold CSV.
    folds = dict(base_folds)
    folds.update({study_id: 5 for study_id in added_ids})
    if len(folds) != 4349 or sum(fold != 0 for fold in folds.values()) != 3524:
        raise ValueError("Expanded split must contain exactly 3,524 train / 825 validation studies")

    with gold_split_path.open(newline="", encoding="utf-8") as stream:
        gold_ids = {row["StudyInstanceUID"] for row in csv.DictReader(stream)}
    if len(gold_ids) != 58 or (gold_ids & label_ids) or (gold_ids & set(folds)):
        raise ValueError("Gold exclusion lock failed")

    # Construct a validation-only Steven-v4 file. Gold labels are never loaded.
    validation_rows = []
    seen = set()
    with steven_csv.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not set(TARGETS + ["StudyInstanceUID"]).issubset(reader.fieldnames or []):
            raise ValueError("Steven-v4 columns do not match the fixed 12-target schema")
        for row in reader:
            study_id = row["StudyInstanceUID"]
            if study_id not in validation_ids:
                continue
            if study_id in seen:
                raise ValueError(f"Duplicate Steven-v4 validation row: {study_id}")
            seen.add(study_id)
            probs, weights = [], []
            for target in TARGETS:
                try:
                    value = float(row[target])
                except (TypeError, ValueError):
                    value = float("nan")
                valid = value == value and 0.0 <= value <= 1.0
                probs.append(value if valid else None)
                weights.append(float(valid))
            validation_rows.append({"study_id": study_id, "probabilities": probs, "weights": weights})
    if seen != validation_ids:
        raise ValueError(f"Steven-v4 validation coverage mismatch: {len(validation_ids - seen)} missing")

    work = Path("/kaggle/working/hpo_v31_trainval_resnet50_fresh")
    run_dir = work / "run"
    run_dir.mkdir(parents=True, exist_ok=True)
    folds_path = work / "report_folds_hpo_add281_trainonly.csv"
    with folds_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(("study_id", "fold"))
        writer.writerows(sorted(folds.items()))
    validation_path = work / "steven_v4_fold0_reference.json"
    validation_path.write_text(json.dumps({
        "version": "steven_v4_fold0_validation_only",
        "targets_in_order": TARGETS,
        "labels": validation_rows,
    }, allow_nan=False) + "\n", encoding="utf-8")
    config = json.loads(base_config_path.read_text(encoding="utf-8"))
    if config["model"].get("backbone") != "resnet34" or config["batch_studies"] != 8:
        raise ValueError("Unexpected reference backbone or batch size")
    config["model"]["backbone"] = "resnet50"
    config["version"] = "hpo_v31_trainval_resnet50_fresh"
    config = {'version': 'hpo_v31_trainval_resnet50_fresh', 'cache': {'field_of_view_mm': 140.0, 'image_size': 224, 'series_slots': 4, 'slices_per_slot': 12, 'central_band': [0.2, 0.8], 'clip_percentiles': [1.0, 99.0], 'exclude_order_sources': ['unordered']}, 'model': {'backbone': 'resnet50', 'pretrained': True, 'dropout': 0.4, 'pooling': 'target_topk_mean', 'top_k_fraction': 0.1}, 'epochs': 60, 'batch_studies': 8, 'learning_rate': 0.0003, 'weight_decay': 0.03, 'warmup_fraction': 0.05, 'gradient_clip_norm': 1.0, 'mixed_precision': True, 'validation_fold': 0, 'xla_eval_slices': 48, 'confident_low': 0.3, 'confident_high': 0.7, 'seed': 2026, 'note': 'Same ResNet-34 reader and plateau-60 recipe as the 0.906 reference. Training uses HPO v1 blend labels; held-out report validation is scored against Steven v4. TPU/XLA uses fixed 24 training and 48 evaluation windows per study; repeated study padding is masked from the loss.', 'early_stopping_patience': 8, 'early_stopping_min_delta': 0.0005, 'early_stopping_metric': 'auc', 'scheduler': {'type': 'reduce_on_plateau', 'mode': 'min', 'factor': 0.5, 'patience': 2, 'threshold': 0.0005, 'threshold_mode': 'abs', 'cooldown': 1, 'min_lr': 1e-06}, 'augmentation': {'crop_area_min': 1.0, 'translate_fraction': 0.06, 'rotation_degrees': 0.0, 'intensity_jitter': 0.15, 'brightness_shift': 0.1, 'erase_probability': 0.25, 'erase_fraction': 0.25}}
    config["note"] = "v3.1-D2 training and validation; fixed fold0; Steven-v4 audit only"
    config_path = work / "cnn_hpo_v31_trainval_resnet50.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")

    if not torch.cuda.is_available():
        raise RuntimeError("GPU is required; this Kaggle run is configured for GPU training")
    torch.cuda.set_device(0)
    gpu_count = torch.cuda.device_count()
    print(f"Accelerator: {gpu_count} GPU(s); primary device {torch.cuda.get_device_name(0)}", flush=True)

    # Match the original single-GPU execution.
    sys.path.insert(0, str(source))
    instrumented = trainer_path.read_text(encoding="utf-8")
    if hashlib.sha256(trainer_path.read_bytes()).hexdigest() != "b8f8302319afbcf9970f0cf5b806d8fb78e72ef6e169b276274ddf7cd5e0ba85":
        raise RuntimeError("Attached trainer differs from locally verified source")
    boundary = '\n    summary = {'
    if instrumented.count(boundary) != 1:
        raise RuntimeError("Trainer summary boundary differs from the verified source")
    instrumented = instrumented.replace(boundary,
        '\n        print(f"CHECKPOINTS_READY epoch={epoch}", flush=True)\n'
        '        if time.monotonic() - KAGGLE_RUN_STARTED + record["seconds"] + 300 > 10.5 * 3600:\n'
        '            print("TIME_BUDGET_STOP: saved epoch; finishing before Kaggle runtime limit", flush=True)\n'
        '            break\n\n    summary = {')
    evaluation_boundary = "        metrics, scores = evaluate()"
    if instrumented.count(evaluation_boundary) != 1:
        raise RuntimeError("Verified evaluation boundary changed")
    instrumented = instrumented.replace(evaluation_boundary,
        evaluation_boundary + "\n        metrics.update(AUDIT_STEVEN(scores, validation_ids, epoch))")
    history_boundary = "        history.append(record)"
    if instrumented.count(history_boundary) != 1:
        raise RuntimeError("Verified history boundary changed")
    instrumented = instrumented.replace(history_boundary,
        '        record["steven_validation_macro_auc"] = metrics["steven_validation_macro_auc"]\n'
        '        record["steven_validation_bce"] = metrics["steven_validation_bce"]\n' + history_boundary)
    instrumented_path = work / "train_cnn_baseline_instrumented.py"
    instrumented_path.write_text(instrumented, encoding="utf-8")
    spec = importlib.util.spec_from_file_location("train_cnn_baseline", instrumented_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import trainer: {trainer_path}")
    trainer = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = trainer
    spec.loader.exec_module(trainer)
    def audit_steven(scores, study_ids, epoch):
        np = trainer.np
        references = {row["study_id"]: row for row in validation_rows}
        target = np.asarray([[0.5 if p is None else p for p in references[s]["probabilities"]]
                             for s in study_ids], dtype=np.float32)
        weight = np.asarray([references[s]["weights"] for s in study_ids], dtype=np.float32)
        metrics = trainer.confident_auc(target, weight, scores, config["confident_low"], config["confident_high"])
        p = np.clip(scores.astype(np.float64), 1e-7, 1-1e-7)
        bce = float(((-target*np.log(p)-(1-target)*np.log1p(-p))*weight).sum()/weight.sum())
        output = {"epoch": epoch, "reference": "Steven v4 audit only", "metrics": metrics,
                  "validation_bce": bce, "study_ids": study_ids, "scores": scores.tolist()}
        (run_dir / f"validation_audit_epoch_{epoch:02d}.json").write_text(json.dumps(output)+"\n")
        print(f"STEVEN_AUDIT epoch={epoch} AUC={metrics['macro_auc']:.6f} BCE={bce:.6f} (not used for controls)", flush=True)
        return {"steven_validation_macro_auc": metrics["macro_auc"], "steven_validation_bce": bce}
    trainer.AUDIT_STEVEN = audit_steven
    trainer.KAGGLE_RUN_STARTED = time.monotonic()

    print("Single-GPU baseline: no DataParallel; global batch 8.", flush=True)

    print("Experiment: HPO-v3.1-D2 provenance labels; ResNet-50/ImageNet; 24 ordered windows; batch 8.", flush=True)
    print("Split: 3,524 train (includes 281 train-only studies), fixed 825 v3.1-D2 validation (Steven-v4 logged only); 58 gold excluded.", flush=True)
    print("Controls: max 60 epochs; AUC checkpoint/early stop (patience 8, min delta 0.0005); BCE ReduceLROnPlateau.", flush=True)
    print("Checkpoints/history are saved to /kaggle/working after every epoch.", flush=True)
    print(f"Inputs: labels={labels_path}; source={source}; cache_shards={len(cache_dirs)}", flush=True)
    print(f"Output: {run_dir}", flush=True)

    sys.argv = [
        str(trainer_path), "--cache", *map(str, cache_dirs),
        "--soft-labels", str(labels_path),
        "--validation-soft-labels", str(labels_path),
        "--folds", str(folds_path),
        "--config", str(config_path),
        "--output-dir", str(run_dir),
        "--validation-fold", "0",
        "--train-slices-per-study", "24",
    ]
    trainer.main()

    summary_path = run_dir / "run_summary.json"
    if not summary_path.is_file():
        raise RuntimeError("Training ended without run_summary.json")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("FINAL_RUN_SUMMARY " + json.dumps({
        key: summary.get(key) for key in
        ("device", "train_studies", "validation_studies", "best", "epochs_run", "stopped_early", "seconds")
    }), flush=True)




In [ ]:
print("STARTUP: definitions imported; resolving fixed inputs and checksum/split locks before training", flush=True)
main()
